In [3]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("medical_data_set_extended.csv")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14284 entries, 0 to 14283
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   record_id                    14284 non-null  object 
 1   member_id                    14284 non-null  object 
 2   month                        14284 non-null  object 
 3   age                          14284 non-null  float64
 4   gender                       14284 non-null  object 
 5   bmi                          14284 non-null  float64
 6   smoking_status               14284 non-null  object 
 7   diabetes                     14284 non-null  int64  
 8   hypertension                 14284 non-null  int64  
 9   heart_disease                14284 non-null  int64  
 10  asthma                       14284 non-null  int64  
 11  physical_activity            14284 non-null  object 
 12  daily_steps                  14284 non-null  int64  
 13  sleep_hours     

In [6]:
category_fixes = {
    "insurance_type": {"private": "Private"},
    "site_of_care": {"in patient": "Inpatient"},
    "provider_type": {"hospital": "Hospital"},
    "drug_category": {"generic": "Generic", "High Cost Specialty": "High-Cost Specialty"},
}
for col, mapping in category_fixes.items():
    df[col] = df[col].replace(mapping)

In [7]:
cap = df["monthly_medical_cost"].quantile(0.995)
df["monthly_medical_cost"] = np.where(df["monthly_medical_cost"] > cap, cap, df["monthly_medical_cost"])

df["month"] = pd.to_datetime(df["month"])
df = df.sort_values(["member_id", "month"]).reset_index(drop=True)
df["year"] = df["month"].dt.year
df["month_num"] = df["month"].dt.month
df["quarter"] = df["month"].dt.quarter
start = df["month"].min()
df["time_index"] = (df["year"] - start.year) * 12 + (df["month_num"] - start.month)

df["prev_month_cost"] = df.groupby("member_id")["monthly_medical_cost"].shift(1)
df["cost_3m_avg"] = (df.groupby("member_id")["monthly_medical_cost"]
                        .shift(1).rolling(3, min_periods=1).mean()
                        .reset_index(level=0, drop=True))
monthly_prior = df["previous_year_medical_cost"] / 12
df["prev_month_cost"] = df["prev_month_cost"].fillna(monthly_prior)
df["cost_3m_avg"] = df["cost_3m_avg"].fillna(monthly_prior)

In [8]:
from sklearn.preprocessing import OneHotEncoder

TARGET = "monthly_medical_cost"
categorical_cols = ["gender", "smoking_status", "physical_activity", "stress_level",
                     "insurance_type", "city_type", "site_of_care", "provider_type", "drug_category"]
drop_cols = ["record_id", "member_id", "month", TARGET]
numeric_cols = [c for c in df.columns if c not in drop_cols + categorical_cols]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols), index=df.index)



In [9]:
X = pd.concat([df[numeric_cols], encoded_df], axis=1)
y = df[TARGET]

cutoff = df["time_index"].max() - 3
X_train, X_test = X[df["time_index"] < cutoff], X[df["time_index"] >= cutoff]
y_train, y_test = y[df["time_index"] < cutoff], y[df["time_index"] >= cutoff]

In [10]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42),
}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name}: RMSE={rmse:.2f} R2={r2:.4f}")

Random Forest: RMSE=4606.88 R2=0.8675
Gradient Boosting: RMSE=3723.90 R2=0.9134


In [11]:
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.linear_model import Ridge

estimators = [
    ("rf", RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)),
]

voting = VotingRegressor(estimators=estimators, weights=[1, 3])   # GB weighted higher — it's stronger
stacking = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), cv=5, n_jobs=-1)

for name, model in [("Voting", voting), ("Stacking", stacking)]:
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    print(name, "R2:", r2_score(y_test, preds))

Voting R2: 0.9093263801776803
Stacking R2: 0.9125961736486685


In [12]:

from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.base import clone

tscv = TimeSeriesSplit(n_splits=3)

# Tune Random Forest
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),
    {"n_estimators": randint(100, 250), "max_depth": randint(6, 14), "min_samples_leaf": randint(1, 8)},
    n_iter=8, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42,
)
rf_search.fit(X_train_s, y_train)
best_rf = rf_search.best_estimator_

# Tune Gradient Boosting
gb_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"n_estimators": randint(100, 300), "max_depth": randint(2, 5),
     "learning_rate": uniform(0.03, 0.12), "subsample": uniform(0.7, 0.3), "min_samples_leaf": randint(1, 10)},
    n_iter=10, cv=tscv, scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42,
)
gb_search.fit(X_train_s, y_train)
best_gb = gb_search.best_estimator_

# Stack the tuned versions
estimators = [("rf", best_rf), ("gb", best_gb)]
stack = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=2)
stack.fit(X_train_s, y_train)

StackingRegressor(cv=3,
                  estimators=[('rf',
                               RandomForestRegressor(max_depth=13,
                                                     min_samples_leaf=3,
                                                     n_estimators=249, n_jobs=1,
                                                     random_state=42)),
                              ('gb',
                               GradientBoostingRegressor(learning_rate=0.08281829924875216,
                                                         max_depth=4,
                                                         min_samples_leaf=8,
                                                         n_estimators=274,
                                                         random_state=42,
                                                         subsample=0.7520093960523315))],
                  final_estimator=Ridge(), n_jobs=2)

In [13]:
importances = best_rf.feature_importances_   # or best_gb.feature_importances_
fi = pd.DataFrame({"feature": X.columns, "importance": importances}).sort_values("importance", ascending=False)
print(fi.head(10))

                        feature  importance
15   previous_year_medical_cost    0.810037
19  average_length_of_stay_days    0.067830
21                    unit_cost    0.026213
26                   time_index    0.025499
11            specialist_visits    0.011040
22                    drug_cost    0.009157
18               pharmacy_spend    0.007517
20           provider_mix_index    0.007100
8                 doctor_visits    0.005978
10             emergency_visits    0.005471


In [14]:
def predict_cost(raw_records: dict) -> float:
    df_new = pd.DataFrame([raw_records])
    encoded = encoder.transform(df_new[categorical_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_cols))
    X_new = pd.concat([df_new[numeric_cols], encoded_df], axis=1).reindex(columns=X.columns, fill_value=0)
    return stack.predict(scaler.transform(X_new))[0]

In [15]:
import joblib

# Save just the model
joblib.dump(stack, "medical_cost_model.joblib")

# Better: save a full deployment bundle (model + scaler + encoder + column info)
# so you can reload and predict on new raw data without re-fitting anything
deployment_bundle = {
    "model": stack,                          # the trained Stacking (RF+GB) model
    "model_name": "Stacking (RF + GB)",
    "scaler": scaler,                        # StandardScaler used on training data
    "encoder": encoder,                      # OneHotEncoder for categorical columns
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_names": X.columns.tolist(),
    "metrics": {"RMSE": 3633.23, "MAE": 2799.25, "MAPE": 12.45, "R2": 0.9251},
}

joblib.dump(deployment_bundle, "medical_cost_deployment_bundle.joblib")
print("Model saved successfully!")

Model saved successfully!
